In [6]:
import json
import os
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV

In [7]:
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / ".git").exists():
        ROOT = _p
        break
os.chdir(ROOT)
print(f"Project root: {ROOT}")

sys.path.insert(0, str(ROOT))
from src.encoding import encode_categoricals

Project root: /Users/alinapopov/Desktop/04_GitHub/dc-mof-project


In [8]:
# ── Config ───────────────────────────────────────────────────────────────────
FEATURE_SET = "geo_decorr"          # geometric-only, decorrelated
STRATEGIES  = ["random", "topology", "metal"]
SPLITS      = ROOT / "data/splits"
MODELS_DIR  = ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

feat_info    = json.loads((ROOT / "data/merged/feature_selection.json").read_text())
feature_cols = feat_info["feature_sets"][FEATURE_SET]
target_cols  = feat_info["target_cols"]   # 5 CO2 pressure points
print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Targets  ({len(target_cols)}): {target_cols}")

Features (5): ['pld', 'surface_area_m2g', 'surface_area_m2cm3', 'void_fraction', 'density_g_cm3']
Targets  (5): ['co2_mol_kg_0.01bar', 'co2_mol_kg_0.05bar', 'co2_mol_kg_0.1bar', 'co2_mol_kg_0.5bar', 'co2_mol_kg_2.5bar']


In [9]:
def train_rf(X_train, y_train):
    """RandomizedSearchCV over RF hyperparams; returns best estimator and params."""
    param_dist = {
        "n_estimators":      [100, 200, 300, 500],
        "max_depth":         [None, 10, 20, 30],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf":  [1, 2, 4],
        "max_features":      ["sqrt", "log2", 0.5],
    }
    search = RandomizedSearchCV(
        RandomForestRegressor(n_jobs=-1, random_state=42),
        param_dist,
        n_iter=20,
        cv=5,
        scoring="r2",
        n_jobs=-1,
        random_state=42,
        refit=True,
        verbose=1,
    )
    search.fit(X_train, y_train)
    return search.best_estimator_, search.best_params_

In [18]:
# ── Train across all 3 splits ────────────────────────────────────────────────
all_metrics = []

for strategy in STRATEGIES:
    print('=' * 60)
    print('Split strategy:', strategy)
    print('=' * 60)

    train = pd.read_parquet(SPLITS / f"split_{strategy}_train_merged.parquet")
    val   = pd.read_parquet(SPLITS / f"split_{strategy}_val_merged.parquet")
    test  = pd.read_parquet(SPLITS / f"split_{strategy}_test_merged.parquet")

    train_enc, val_enc, test_enc = encode_categoricals(
        train, val, test, method="label", unknown_int="n_classes"
    )

    train_med = train_enc[feature_cols].median()
    X_train = train_enc[feature_cols].fillna(train_med)
    y_train = train_enc[target_cols]
    X_val   = val_enc[feature_cols].fillna(train_med)
    y_val   = val_enc[target_cols]
    X_test  = test_enc[feature_cols].fillna(train_med)
    y_test  = test_enc[target_cols]

    model, best_params = train_rf(X_train, y_train)
    print(f"Best params: {best_params}")

    for split_name, X, y in [("val", X_val, y_val), ("test", X_test, y_test)]:
        preds = model.predict(X)
        r2   = r2_score(y, preds, multioutput="uniform_average")
        rmse = mean_squared_error(y, preds, multioutput="uniform_average") ** 0.5
        print(f"  {split_name:4s}  R²={r2:.4f}  RMSE={rmse:.4f}")
        all_metrics.append({
            "strategy": strategy, "split": split_name,
            "feature_set": FEATURE_SET, "r2": round(r2, 4), "rmse": round(rmse, 4),
            **best_params,
        })

    model_path = MODELS_DIR / f"rf_{FEATURE_SET}_{strategy}.joblib"
    joblib.dump(model, model_path)
    print(f"  saved -> {model_path.relative_to(ROOT)}")

SyntaxError: unterminated string literal (detected at line 5) (3570351905.py, line 5)

In [ ]:
# ── Save metrics log ─────────────────────────────────────────────────────────
metrics_df   = pd.DataFrame(all_metrics)
metrics_path = MODELS_DIR / f"metrics_{FEATURE_SET}.csv"
metrics_df.to_csv(metrics_path, index=False)
print(metrics_df[["strategy", "split", "r2", "rmse"]].to_string(index=False))
print('Metrics saved ->', metrics_path.relative_to(ROOT))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

metrics_df = pd.read_csv(MODELS_DIR / ('metrics_' + FEATURE_SET + '.csv'))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
strategies = ['random', 'topology', 'metal']
splits     = ['val', 'test']
x          = range(len(strategies))
width      = 0.35

for ax, metric in zip(axes, ['r2', 'rmse']):
    for i, split in enumerate(splits):
        vals = [
            metrics_df.loc[(metrics_df['strategy'] == s) & (metrics_df['split'] == split), metric].values[0]
            for s in strategies
        ]
        offset = (i - 0.5) * width
        bars = ax.bar([xi + offset for xi in x], vals, width, label=split)
        ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(strategies)
    ax.set_xlabel('Split strategy')
    ax.set_ylabel(metric.upper())
    ax.set_title(metric.upper() + ' by split strategy (' + FEATURE_SET + ')')
    ax.legend()
    if metric == 'r2':
        ax.set_ylim(0, 1.05)

plt.tight_layout()
plot_path = MODELS_DIR / ('plot_' + FEATURE_SET + '.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print('Plot saved ->', plot_path.relative_to(ROOT))